# Atlas example — экзопланеты → перебор → размерность $G$

Этот ноутбук показывает **полностью воспроизводимый локальный пример** обратного dimensional-closure пути:

$$\text{данные} \rightarrow \text{целочисленный перебор степенных замыканий} \rightarrow \dim C \rightarrow \text{exact }\ker D.$$

**Граница утверждения.** Это не новый закон природы и не повтор исторического 172×132 exoplanet receipt. В текущем дистрибутиве сохранилось описание того опыта, но исходная 172-строчная таблица отсутствует. Поэтому здесь запечатан отдельный реальный ретроспективный snapshot: 50 строк NASA Exoplanet Archive (export 2018-03-03), в которых одновременно заданы orbital period, semi-major axis и stellar mass. Данные выбираются **до** перебора только по наличию этих трёх положительных измерений.

Поиск **не получает ни значение $G$, ни его размерность, ни закон Кеплера**. После freeze победителя локальный registry открывается только для post-hoc распознавания размерностного класса. Полный inverse Dimensional Closure v2.0 остаётся research workflow, а не отдельным production owner; exact dimensional kernel берётся из текущего Atlas core.

In [1]:
from pathlib import Path
import csv, itertools, json, math, sys
from fractions import Fraction
import numpy as np

# Находим корень release независимо от того, запускается notebook из root или examples/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "source").is_dir() and (p / "data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Atlas release root not found")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "examples" / "data" / "exoplanets_g_dimension_nasa2018.csv"
print("release root resolved: OK")
print("data:", DATA.relative_to(ROOT))

release root resolved: OK
data: examples/data/exoplanets_g_dimension_nasa2018.csv


## 1. Загружаем запечатанные данные экзопланет

In [2]:
rows = []
with DATA.open(encoding="utf-8", newline="") as f:
    for r in csv.DictReader(f):
        rows.append({
            "source_rowid": int(r["source_rowid"]),
            "planet": r["planet_name"],
            "host": r["host_name"],
            "P_days": float(r["pl_orbper_days"]),
            "a_AU": float(r["pl_orbsmax_au"]),
            "M_solar": float(r["st_mass_solar"]),
        })

assert rows and all(r["P_days"] > 0 and r["a_AU"] > 0 and r["M_solar"] > 0 for r in rows)
print("rows:", len(rows))
print("hosts:", len({r['host'] for r in rows}))
print("first 5:")
for r in rows[:5]:
    print(r)

rows: 50
hosts: 38
first 5:
{'source_rowid': 1, 'planet': '11 Com b', 'host': '11 Com', 'P_days': 326.03, 'a_AU': 1.29, 'M_solar': 2.7}
{'source_rowid': 2, 'planet': '11 UMi b', 'host': '11 UMi', 'P_days': 516.22, 'a_AU': 1.54, 'M_solar': 1.8}
{'source_rowid': 3, 'planet': '14 And b', 'host': '14 And', 'P_days': 185.84, 'a_AU': 0.83, 'M_solar': 2.2}
{'source_rowid': 4, 'planet': '14 Her b', 'host': '14 Her', 'P_days': 1773.4, 'a_AU': 2.77, 'M_solar': 0.9}
{'source_rowid': 5, 'planet': '16 Cyg B b', 'host': '16 Cyg B', 'P_days': 798.5, 'a_AU': 1.681, 'M_solar': 0.99}


## 2. Переводим только единицы — физический закон не подсказываем

Перебор видит три наблюдаемых величины:

- $a$ — semi-major axis, размерность $L$;
- $P$ — orbital period, размерность $T$;
- $M_\star$ — stellar mass, размерность $M$.

И ищет primitive integer closure

$$C_i=a_i^{e_a}P_i^{e_P}M_{\star,i}^{e_M},$$

для всех допустимых троек $(e_a,e_P,e_M)$ в заранее фиксированной оболочке. Чем меньше разброс $\ln C_i$, тем сильнее наблюдаемое замыкание. Нормированная метрика используется, чтобы сравнивать разные степенные направления.

In [3]:
AU_M = 149_597_870_700.0
DAY_S = 86_400.0
M_SUN_KG = 1.98847e30

a = np.array([r["a_AU"] for r in rows], float) * AU_M
P = np.array([r["P_days"] for r in rows], float) * DAY_S
M = np.array([r["M_solar"] for r in rows], float) * M_SUN_KG
LX = np.column_stack([np.log(a), np.log(P), np.log(M)])

SEARCH_BOUNDS = {"e_a": (-4, 4), "e_P": (-4, 4), "e_M": (-3, 3)}
print(SEARCH_BOUNDS)

{'e_a': (-4, 4), 'e_P': (-4, 4), 'e_M': (-3, 3)}


## 3. Полный перебор фиксированной оболочки

In [4]:
def primitive(v):
    g = 0
    for x in v:
        g = math.gcd(g, abs(int(x)))
    return g == 1

def canonical_sign(v):
    first = next((x for x in v if x), 1)
    return first > 0

def closure_score(v):
    v = np.asarray(v, int)
    lnC = LX @ v
    raw = float(np.std(lnC, ddof=1))
    scale = math.sqrt(sum((int(v[j]) * float(np.std(LX[:, j], ddof=1)))**2 for j in range(3)))
    return raw / scale, raw, float(np.exp(np.mean(lnC)))

candidates = []
for ea, eP, eM in itertools.product(range(-4, 5), range(-4, 5), range(-3, 4)):
    v = (ea, eP, eM)
    # tri-variable closure lane: все три наблюдаемые должны участвовать;
    # ±v и кратные векторы не считаются разными гипотезами.
    if 0 in v or not primitive(v) or not canonical_sign(v):
        continue
    rho, raw, Chat = closure_score(v)
    candidates.append({"exponents": v, "rho": rho, "sd_logC": raw, "C_hat_SI": Chat})

candidates.sort(key=lambda x: (x["rho"], sum(abs(e) for e in x["exponents"]), x["exponents"]))
print("structural candidates evaluated:", len(candidates))
print("top 10:")
for rank, c in enumerate(candidates[:10], 1):
    print(f"{rank:2d}  e={c['exponents']}  rho={c['rho']:.6f}  sd(log C)={c['sd_logC']:.6f}")

winner = dict(candidates[0])  # freeze ДО чтения registry constants
assert winner["exponents"] == (3, -2, -1)
print("\nFROZEN WINNER:", winner)

structural candidates evaluated: 172
top 10:
 1  e=(3, -2, -1)  rho=0.008062  sd(log C)=0.048328
 2  e=(3, -2, -2)  rho=0.075289  sd(log C)=0.454899
 3  e=(4, -3, -1)  rho=0.076833  sd(log C)=0.651354
 4  e=(4, -3, -2)  rho=0.097172  sd(log C)=0.827043
 5  e=(4, -3, 1)  rho=0.120663  sd(log C)=1.022926
 6  e=(4, -3, -3)  rho=0.134329  sd(log C)=1.150769
 7  e=(3, -2, 1)  rho=0.142990  sd(log C)=0.857165
 8  e=(3, -2, -3)  rho=0.145416  sd(log C)=0.889978
 9  e=(4, -3, 2)  rho=0.163152  sd(log C)=1.388607
10  e=(2, -1, -1)  rho=0.188361  sd(log C)=0.672013

FROZEN WINNER: {'exponents': (3, -2, -1), 'rho': 0.008061939711222455, 'sd_logC': 0.04832792485093855, 'C_hat_SI': 1.731527169756757e-12}


Победитель порождён данными как

$$C=\frac{a^3}{P^2M_\star}.$$

На этом этапе символа $G$ ещё нет. Теперь Atlas может вывести **размерность отсутствующей оси** только из размерностей измеряемых величин.

In [5]:
BASIS = ("L", "M", "T", "I", "Theta", "N", "J")
DIM_A = np.array([1,0,0,0,0,0,0], int)
DIM_P = np.array([0,0,1,0,0,0,0], int)
DIM_M = np.array([0,1,0,0,0,0,0], int)
ea, eP, eM = winner["exponents"]
dim_C = ea*DIM_A + eP*DIM_P + eM*DIM_M

def dim_text(d):
    return " ".join(f"{b}^{int(x)}" for b, x in zip(BASIS, d) if x) or "1"

print("generated missing-axis dimension vector:", tuple(int(x) for x in dim_C))
print("generated missing-axis dimension:", dim_text(dim_C))
assert tuple(dim_C) == (3,-1,-2,0,0,0,0)

generated missing-axis dimension vector: (3, -1, -2, 0, 0, 0, 0)
generated missing-axis dimension: L^3 M^-1 T^-2


## 4. Возвращаем порождённую ось в exact dimensional kernel Atlas

In [6]:
# Тот же exact rational null-space kernel, который использует current dimensional owner.
from source.phi_compiler_owner import _fraction_nullspace, _fraction_rref

axes = [tuple(DIM_A), tuple(DIM_P), tuple(DIM_M), tuple(int(x) for x in dim_C)]
D = [[int(axes[j][i]) for j in range(len(axes))] for i in range(7)]
_rref, pivots = _fraction_rref(D)
ns = _fraction_nullspace(D)

print("rank(D):", len(pivots))
print("p = n-rank(D):", len(axes)-len(pivots))
print("exact ker(D):", ns)
assert len(ns) == 1

# Нормируем знак для читаемой записи Pi.
v = [Fraction(x) for x in ns[0]]
if v[0] < 0:
    v = [-x for x in v]
print("Pi exponents [a, P, M_star, C_generated]:", tuple(int(x) for x in v))
print("Pi = a^3 * P^-2 * M_star^-1 * C_generated^-1 = 1")

rank(D): 3
p = n-rank(D): 1
exact ker(D): ((Fraction(-3, 1), Fraction(2, 1), Fraction(1, 1), Fraction(1, 1)),)
Pi exponents [a, P, M_star, C_generated]: (3, -2, -1, -1)
Pi = a^3 * P^-2 * M_star^-1 * C_generated^-1 = 1


## 5. Только после freeze: распознаём размерностный класс в registry

Теперь разрешён post-hoc lookup. Совпадение размерности с `CONST-G` означает, что система породила ось класса

$$[G]=L^3M^{-1}T^{-2}.$$

Имя `G` и CODATA value **не участвовали** в переборе. Численное сравнение ниже также post-hoc. Для ньютоновской двухтельной нормировки $a^3/(P^2M)=G/(4\pi^2)$, поэтому множитель $4\pi^2$ применяется только после freeze для интерпретации масштаба.

In [7]:
registry = json.loads((ROOT / "data" / "constants" / "registry.json").read_text(encoding="utf-8"))
items = registry if isinstance(registry, list) else registry.get("constants", registry.get("items", []))
G = next(x for x in items if x.get("constant_id") == "CONST-G")
reg_dim = G["dimension"]
G_dim = (
    int(reg_dim["length"]), int(reg_dim["mass"]), int(reg_dim["time"]),
    int(reg_dim["current"]), int(reg_dim["temperature"]), int(reg_dim["amount"]), int(reg_dim["luminous_intensity"])
)
assert G_dim == tuple(int(x) for x in dim_C)

C_hat = winner["C_hat_SI"]
G_hat = 4 * math.pi**2 * C_hat
G_ref = float(G["value"])
print("registry match:", G["constant_id"], G["symbol"], G["unit"])
print("generated dimension == registry G dimension:", G_dim == tuple(dim_C))
print(f"C_hat = {C_hat:.8e} SI")
print(f"post-hoc G_hat = 4*pi^2*C_hat = {G_hat:.8e}")
print(f"registry G = {G_ref:.8e}")
print(f"relative difference = {(G_hat/G_ref-1)*100:.3f}%")

registry match: CONST-G G m^3 kg^-1 s^-2
generated dimension == registry G dimension: True
C_hat = 1.73152717e-12 SI
post-hoc G_hat = 4*pi^2*C_hat = 6.83579527e-11
registry G = 6.67430000e-11
relative difference = 2.420%


## 6. Cross-system sanity check и итог

In [8]:
# Четыре группы по stellar-mass quartiles; это diagnostic, не promotion gate.
lnC = LX @ np.asarray(winner["exponents"], int)
q = np.quantile(M, [0, .25, .5, .75, 1])
group = np.digitize(M, q[1:-1], right=True)
group_means = [float(np.mean(lnC[group == g])) for g in range(4)]
between_group_dispersion = float(np.std(group_means, ddof=1))

summary = {
    "status": "RETROSPECTIVE_DIMENSIONAL_CLOSURE_EXAMPLE_PASS",
    "rows": len(rows),
    "hosts": len({r['host'] for r in rows}),
    "candidate_count": len(candidates),
    "frozen_exponents": winner["exponents"],
    "generated_dimension": tuple(int(x) for x in dim_C),
    "generated_dimension_text": dim_text(dim_C),
    "exact_nullity_after_axis_birth": len(ns),
    "between_group_logC_dispersion": between_group_dispersion,
    "registry_dimension_match_CONST_G": G_dim == tuple(dim_C),
    "world_law_discovery_claimed": False,
    "historical_172_row_receipt_reproduced": False,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary["status"].endswith("PASS")
assert summary["candidate_count"] == 172
assert summary["registry_dimension_match_CONST_G"] is True

{
  "status": "RETROSPECTIVE_DIMENSIONAL_CLOSURE_EXAMPLE_PASS",
  "rows": 50,
  "hosts": 38,
  "candidate_count": 172,
  "frozen_exponents": [
    3,
    -2,
    -1
  ],
  "generated_dimension": [
    3,
    -1,
    -2,
    0,
    0,
    0,
    0
  ],
  "generated_dimension_text": "L^3 M^-1 T^-2",
  "exact_nullity_after_axis_birth": 1,
  "between_group_logC_dispersion": 0.014356233158234628,
  "registry_dimension_match_CONST_G": true,
  "world_law_discovery_claimed": false,
  "historical_172_row_receipt_reproduced": false
}


### Что именно показано

1. Реальные exoplanet measurements загружены из локального sealed CSV.
2. Перебор 172 primitive tri-variable exponent hypotheses выполняется без доступа к `CONST-G`.
3. Лучшее замыкание freeze-ится как `(3, -2, -1)`.
4. Из него рождается missing-axis dimension `L^3 M^-1 T^-2`.
5. После добавления оси exact `ker D` имеет `p=1`.
6. Только затем registry узнаёт тот же размерностный класс как `G`.

Это демонстрирует механизм **порождения размерности**, а не доказывает независимое открытие гравитационной постоянной: semi-major axes и stellar masses в каталогах могут быть модельно связанными, snapshot ретроспективный, а historical 172-row preregistered table в current distribution отсутствует.